# 04 提出サンプルと集計層の EDA

提供 ZIP の提出サンプルと変換スクリプトを確認する。サンプルは例示用の予測であり、テストの正解ではない。値は各集計層の重みを掛けた値。全行を使い、6 層の行数・キー・値と日次行からの再集計を検査する。

In [1]:
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
from IPython.display import display
ROOT=Path.cwd()
if not (ROOT / "input").is_dir(): ROOT=ROOT.parent
assert (ROOT / "input").is_dir()
with zipfile.ZipFile(ROOT / "input/submission_scripts.zip") as z:
    print("ZIP contents:", z.namelist())
    with z.open("submission_scripts/sample_submission_weighted.csv") as f:
        sample=pd.read_csv(f, header=None, names=["index","weighted_sales"])
    source=z.read("submission_scripts/convert_submission.py").decode("utf-8")
# 配布スクリプトの定数のみを使い、CLI は起動しない。
namespace={"__name__":"provided_submission_converter"}
exec(compile(source,"convert_submission.py","exec"),namespace)
layers=namespace["LAYERS"]
print("shape:",sample.shape,"missing:",sample.isna().sum().to_dict(),"duplicate index:",sample['index'].duplicated().sum())
print("nonfinite:",int((~np.isfinite(sample.weighted_sales)).sum()),"negative:",int((sample.weighted_sales<0).sum()))
sample["layer"]=sample['index'].str.split('|',n=1).str[0]
display(sample.groupby('layer').weighted_sales.agg(['size','min','median','mean','max']))
print("層別の集計キーと重み:")
display(pd.DataFrame([{'layer':k,'keys':', '.join(v[0]),'weight':v[1]} for k,v in layers.items()]))

ZIP contents: ['submission_scripts/convert_submission.py', 'submission_scripts/convert_submission.ipynb', 'submission_scripts/submit_guide.pdf', 'submission_scripts/sample_submission_weighted.csv']
shape: (352118, 2) missing: {'index': 0, 'weighted_sales': 0} duplicate index: 0
nonfinite: 0 negative: 0
層別の集計キーと重み:


,size,min,median,mean,max
layer,,,,,
daily_item,30660,0.000000,0.966127,1.791950,18.901359
daily_store,3650,11.477736,18.752807,19.242720,27.654372
date_store_item,306600,0.000000,0.222129,0.474512,7.371320
monthly_item,1008,0.000000,5.681925,10.470375,112.558808
monthly_store,120,73.336003,125.894339,133.561436,195.626862
monthly_store_item,10080,0.000000,1.408967,2.992250,47.374659


,layer,keys,weight
0,date_store_item,"date, store_id, item_id",0.000066
1,daily_store,"date, store_id",0.000032
2,daily_item,"date, item_id",0.000025
3,monthly_store_item,"month, store_id, item_id",0.000014
4,monthly_store,"month, store_id",0.000007
5,monthly_item,"month, item_id",0.000005


## サンプルの全層を日次予測から再構成して照合
丸め誤差を考慮し `np.isclose` を使用。基準となる日次層を重みで割って円に戻し、同じ値から他の層を集計する。

In [2]:
base=sample[sample.layer=='date_store_item'].copy()
parts=base['index'].str.split('|',expand=True)
base=pd.DataFrame({'date':parts[1].to_numpy(),'store_id':parts[2].to_numpy(),'item_id':parts[3].to_numpy(),'sales':base.weighted_sales.to_numpy()/layers['date_store_item'][1]})
base['month']=base.date.str.slice(0,7)
print('Base coverage:',len(base),'dates:',base.date.nunique(),'stores:',base.store_id.nunique(),'items:',base.item_id.nunique())
test=pd.read_csv(ROOT/'input/test.csv',usecols=['date','store_id','item_id'])
keys=['date','store_id','item_id']
coverage=test.merge(base[keys],on=keys,how='outer',indicator=True,validate='one_to_one')
print('test key coverage:',coverage._merge.value_counts().to_dict())
checks=[]
for name,(keys,weight) in layers.items():
    agg=base.groupby(keys,as_index=False,sort=True).sales.sum()
    idx=pd.Series(name,index=agg.index)
    for key in keys: idx=idx+'|'+agg[key]
    expected=pd.DataFrame({'index':idx,'expected':agg.sales*weight})
    actual=sample.loc[sample.layer==name,['index','weighted_sales']]
    joined=expected.merge(actual,on='index',how='outer',indicator=True,validate='one_to_one')
    bad=(~np.isclose(joined.expected,joined.weighted_sales,rtol=1e-9,atol=1e-12)) | joined._merge.ne('both')
    checks.append({'layer':name,'expected_rows':len(expected),'sample_rows':len(actual),'unmatched_or_inconsistent':int(bad.sum()),'max_abs_difference':float((joined.expected-joined.weighted_sales).abs().max())})
checks=pd.DataFrame(checks); display(checks)
print('all layers consistent:',bool(checks.unmatched_or_inconsistent.eq(0).all()))

Base coverage: 306600 dates: 365 stores: 10 items: 84
test key coverage: {'both': 306600, 'left_only': 0, 'right_only': 0}
all layers consistent: True


,layer,expected_rows,sample_rows,unmatched_or_inconsistent,max_abs_difference
0,date_store_item,306600,306600,0,5.551115e-17
1,daily_store,3650,3650,0,3.552714e-15
2,daily_item,30660,30660,0,8.881784e-16
3,monthly_store_item,10080,10080,0,3.552714e-15
4,monthly_store,120,120,0,2.842171e-14
5,monthly_item,1008,1008,0,1.421085e-14


## 提出上の注意
- 日×店舗×商品の非負の予測を生成し、配布された変換スクリプトで6層の提出ファイルに変換する。
- このサンプルの値を正解としてモデルの評価に使わない。提出前に公式スクリプトの `--validate` を実行する。